In [1]:
import cv2
import numpy as np
import uvicorn
import io
from fastapi import FastAPI, UploadFile, File, HTTPException
from fastapi.responses import StreamingResponse
from typing import List
from enum import Enum

from fastapi.templating import Jinja2Templates
from fastapi import Request


In [2]:
# Initialize FastAPI app
app = FastAPI(title='YOLOv3 Object Detection API')

In [3]:
# Load YOLOv3 configuration and weights files
net = cv2.dnn.readNet('yolov3.weights', 'yolov3.cfg')

In [4]:
# Load class names
with open('coco.names', 'r') as f:
    classes = f.read().strip().split('\n')

In [5]:
# Enum for supported YOLO models
class Model(str, Enum):
    yolov3 = "yolov3"

In [6]:
def detect_objects(image: np.ndarray, confidence_threshold: float = 0.5, nms_threshold: float = 0.4):
    """Detect objects using YOLOv3 and return the processed image with bounding boxes."""
    
    height, width = image.shape[:2]

    # Create a blob from the image
    blob = cv2.dnn.blobFromImage(image, 1/255.0, (416, 416), swapRB=True, crop=False)
    net.setInput(blob)

    # Get output layer names
    layer_names = net.getLayerNames()
    output_layers = [layer_names[i - 1] for i in net.getUnconnectedOutLayers()]

    # Perform forward pass
    outputs = net.forward(output_layers)

    # Initialize lists for detected bounding boxes, confidences, and class IDs
    boxes = []
    confidences = []
    class_ids = []

    # Iterate over each detection
    for output in outputs:
        for detection in output:
            scores = detection[5:]
            class_id = np.argmax(scores)
            confidence = scores[class_id]

            if confidence > confidence_threshold:
                # Scale bounding box coordinates to the original image size
                center_x = int(detection[0] * width)
                center_y = int(detection[1] * height)
                w = int(detection[2] * width)
                h = int(detection[3] * height)

                # Calculate top-left corner of the bounding box
                x = int(center_x - w / 2)
                y = int(center_y - h / 2)

                # Append to lists
                boxes.append([x, y, w, h])
                confidences.append(float(confidence))
                class_ids.append(class_id)

    # Apply Non-Maximum Suppression
    indices = cv2.dnn.NMSBoxes(boxes, confidences, confidence_threshold, nms_threshold)

    # Draw bounding boxes and labels on the image
    if len(indices) > 0:
        for i in indices.flatten():
            box = boxes[i]
            x, y, w, h = box
            label = str(classes[class_ids[i]])
            confidence = confidences[i]

            # Draw the bounding box
            cv2.rectangle(image, (x, y), (x + w, y + h), (0, 0, 255), 2)

            # Display the label and confidence score
            text = f"{label}: {confidence:.2f}"
            cv2.putText(image, text, (x, y - 10), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2)

    return image


In [7]:
@app.get("/")
def home():
    return "Congratulations! Your YOLOv3 Object Detection API is running."

In [8]:
@app.post("/predict")
def predict(model: Model, file: UploadFile = File(...)):
    # Validate the uploaded file format
    filename = file.filename
    file_extension = filename.split(".")[-1] in ("jpg", "jpeg", "png")
    if not file_extension:
        raise HTTPException(status_code=415, detail="Unsupported file type provided.")

    # Read the image as a stream of bytes
    image_stream = io.BytesIO(file.file.read())
    image_stream.seek(0)

    # Convert the stream to a NumPy array
    file_bytes = np.asarray(bytearray(image_stream.read()), dtype=np.uint8)
    
    # Decode the image
    image = cv2.imdecode(file_bytes, cv2.IMREAD_COLOR)

    # Perform object detection
    processed_image = detect_objects(image)

    # Convert the image back to a byte stream
    _, img_encoded = cv2.imencode('.jpg', processed_image)
    img_bytes = img_encoded.tobytes()

    # Return the image with bounding boxes as a streaming response
    return StreamingResponse(io.BytesIO(img_bytes), media_type="image/jpeg")


In [ ]:
import nest_asyncio
nest_asyncio.apply()


# Run the server
if __name__ == "__main__":
    uvicorn.run(app, host="127.0.0.1", port=8000)

INFO:     Started server process [15212]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)


INFO:     127.0.0.1:51628 - "GET / HTTP/1.1" 200 OK
INFO:     127.0.0.1:51628 - "GET /docs HTTP/1.1" 200 OK
INFO:     127.0.0.1:51628 - "GET /openapi.json HTTP/1.1" 200 OK
